# Review Aspect Analyzer: train and demo on Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spiyush09/review_Aspect_Analyzer/blob/main/notebooks/train_colab.ipynb)

This notebook fine-tunes BERT on SemEval-2014 (restaurant aspect sentiment), evaluates it, and runs the real Streamlit app so you can try it.

1. **Runtime -> Change runtime type -> T4 GPU**, then Save.
2. **Runtime -> Run all** (or run the cells one by one with Shift + Enter). Training takes about 10-15 minutes.
3. The Streamlit section prints a temporary public link. Open it in a new tab to try the app.
   If the link does not open, use the "show results inside Colab" section instead.
4. The last cell downloads `output.zip` (the trained model and results) to use locally.

## Get the project and install packages

In [1]:
%cd /content
!rm -rf review_Aspect_Analyzer
!git clone -q https://github.com/spiyush09/review_Aspect_Analyzer.git
%cd review_Aspect_Analyzer
!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm

/content/review_Aspect_Analyzer
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 104.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 105.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Train the model

In [2]:
!python -m src.train

train=2837  val=367  test=365
config.json: 100% 570/570 [00:00<00:00, 3.22MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 198kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 4.55MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 8.67MB/s]
Map: 100% 2837/2837 [00:00<00:00, 10264.73 examples/s]
Map: 100% 367/367 [00:00<00:00, 9356.25 examples/s]

model.safetensors: downloading bytes:  86% 380M/440M [00:01<00:00, 383MB/s, 32.2MB/s  ]
model.safetensors: reconstructing file:  58% 256M/440M [00:01<00:01, 140MB/s]
model.safetensors: downloading bytes: 100% 415M/415M [00:02<00:00, 166MB/s, 37.9MB/s  ]
model.safetensors: reconstructing file: 100% 440M/440M [00:02<00:00, 176MB/s, 40.2MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 7917.23it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | U

## Check that the model and results were created

In [3]:
!ls models/absa_model
!ls results

config.json	   tokenizer_config.json  training_args.bin
model.safetensors  tokenizer.json
classification_report.txt  confusion_matrix.png  metrics.json  run_info.json


## Start the Streamlit app and get a public link
Wait about 20 seconds. A link ending in `trycloudflare.com` will be printed. Open it in a new tab.
Leave this notebook running while you take the screenshots.

In [4]:
import subprocess, time, re, os

# download cloudflared (gives a temporary public link to the app)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

# start the Streamlit app in the background
subprocess.Popen(
    "streamlit run app/streamlit_app.py --server.port 8501 --server.headless true "
    "--server.enableCORS false --server.enableXsrfProtection false > streamlit.log 2>&1",
    shell=True)
time.sleep(10)

# start the tunnel and read the link from its log
log = open("tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
                 stdout=log, stderr=subprocess.STDOUT)
link = None
for _ in range(40):
    time.sleep(1)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("tunnel.log").read())
    if m:
        link = m.group(0)
        break
print("Open this link:", link if link else "not found - use the next section instead")

Open this link: https://fred-terms-replied-regular.trycloudflare.com


## Alternative: show results inside Colab
Use this if the link above does not work. Change the last review to one you write yourself.

In [5]:
import sys
sys.path.append(".")
import config
from IPython.display import display
from src.aggregate import analyze_review
from src.predict import AspectSentimentPredictor

predictor = AspectSentimentPredictor(config.MODEL_DIR)
colors = {"negative": "#d9534f", "neutral": "#9e9e9e", "positive": "#4caf50"}

reviews = [
    "The food was amazing but the service was really slow and the prices are too high.",
    "The waiter was rude and the soup was cold. Never coming back.",
    "We ordered pasta and a salad. The restaurant is near the station.",
    "WRITE YOUR OWN REVIEW HERE",
]
for r in reviews:
    print("\nReview:", r)
    result = analyze_review(r, predictor)
    display(result.style.map(lambda v: f"color: {colors[v]}; font-weight: 600", subset=["sentiment"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Review: The food was amazing but the service was really slow and the prices are too high.


,aspect,sentiment,confidence,sentence
0,food,positive,0.892000,The food was amazing but the service was really slow and the prices are too high.
1,service,negative,0.852000,The food was amazing but the service was really slow and the prices are too high.
2,price,negative,0.744000,The food was amazing but the service was really slow and the prices are too high.



Review: The waiter was rude and the soup was cold. Never coming back.


,aspect,sentiment,confidence,sentence
0,waiter,negative,0.975000,The waiter was rude and the soup was cold.
1,soup,negative,0.971000,The waiter was rude and the soup was cold.



Review: We ordered pasta and a salad. The restaurant is near the station.


,aspect,sentiment,confidence,sentence
0,pasta,neutral,0.942000,We ordered pasta and a salad.
1,salad,neutral,0.929000,We ordered pasta and a salad.
2,restaurant,neutral,0.864000,The restaurant is near the station.
3,station,neutral,0.602000,The restaurant is near the station.



Review: WRITE YOUR OWN REVIEW HERE


,aspect,sentiment,confidence,sentence
0,review,positive,0.526000,WRITE YOUR OWN REVIEW HERE


In [6]:
from google.colab import files
files.download("results/confusion_matrix.png")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Download the trained model and results
Extract `output.zip` inside your local project folder so you get `models/absa_model/` and `results/`.

In [ ]:
!zip -r output.zip models/ results/

from google.colab import files
files.download("output.zip")